In [ ]:
# ============================================================
# 03_modeling.ipynb
# IoT Traffic Fingerprinting + Rogue Detection — Layers 3, 4, 5
# ============================================================
# This notebook trains and evaluates:
#   Layer 3: Supervised classification (RF, XGBoost)
#   Layer 4: Unsupervised anomaly detection (KMeans, Autoencoder)
#   Layer 5: Decision fusion + risk scoring
#
# Development: set USE_DEV_SAMPLE = True (loads 50K dev sample, free Colab friendly)
# Full runs:    set USE_DEV_SAMPLE = False (loads full train/val/test, Colab Pro required)
#
# Credits: Layer 5 fusion formula, risk thresholds, and alert/action logic
# adapted from Oscar's capstone.ipynb (April 2026). Refactored for proper
# train/val/test separation and locked threshold rule.
# ============================================================

# ====== TOGGLE THIS FOR DEV vs FULL RUNS ======
USE_DEV_SAMPLE = True   # True = dev_sample (50K rows), False = full train/val/test
# ===============================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

# ---------- Paths ----------
BASE = '/content/drive/MyDrive/Capstone_IoT'
PROCESSED = os.path.join(BASE, 'Data/Processed')
RESULTS_L3 = os.path.join(BASE, 'Results/Layer3')
RESULTS_L4 = os.path.join(BASE, 'Results/Layer4')
RESULTS_L5 = os.path.join(BASE, 'Results/Layer5')
for d in [RESULTS_L3, RESULTS_L4, RESULTS_L5]:
    os.makedirs(d, exist_ok=True)

# ---------- Constants ----------
RANDOM_SEED = 42
LABEL_COL = 'Label'
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Mode: {'DEV_SAMPLE (50K rows)' if USE_DEV_SAMPLE else 'FULL DATA (train/val/test)'}")
print(f"Results directories created under {BASE}/Results/")

Mounted at /content/drive
Mode: DEV_SAMPLE (50K rows)
Results directories created under /content/drive/MyDrive/Capstone_IoT/Results/


In [ ]:
# ============================================================
# Cell 2: Load parquet splits
# ============================================================

def load_parquet(name):
    path = os.path.join(PROCESSED, name)
    df = pd.read_parquet(path)
    size_mb = os.path.getsize(path) / (1024**2)
    print(f"  {name:28s} {df.shape}  ({size_mb:.1f} MB)")
    return df

print("Loading parquet splits...")

if USE_DEV_SAMPLE:
    # Development mode: use the 50K dev sample for train,
    # carve small val/test slices from it for fast iteration.
    # Unknown holdout is always loaded in full (it's tiny).
    dev = load_parquet('dev_sample.parquet')
    unknown = load_parquet('unknown_holdout.parquet')

    # Stratified split of dev sample into 70/15/15 for layer-aware dev
    from sklearn.model_selection import train_test_split
    train_val_dev, test = train_test_split(
        dev, test_size=0.15, stratify=dev[LABEL_COL], random_state=RANDOM_SEED
    )
    train, val = train_test_split(
        train_val_dev, test_size=0.1765,  # 0.15 / 0.85
        stratify=train_val_dev[LABEL_COL], random_state=RANDOM_SEED
    )
    train = train.reset_index(drop=True)
    val = val.reset_index(drop=True)
    test = test.reset_index(drop=True)

    print(f"\n[DEV MODE] Sub-split of dev_sample:")
    print(f"  train: {train.shape}")
    print(f"  val:   {val.shape}")
    print(f"  test:  {test.shape}")
else:
    # Full mode: load proper train/val/test from Phase 1
    train = load_parquet('train.parquet')
    val = load_parquet('val.parquet')
    test = load_parquet('test.parquet')
    unknown = load_parquet('unknown_holdout.parquet')

# ---------- Feature / label split ----------
FEATURE_COLS = [c for c in train.columns if c != LABEL_COL]
print(f"\nFeature columns: {len(FEATURE_COLS)}")

X_train = train[FEATURE_COLS].values
X_val   = val[FEATURE_COLS].values
X_test  = test[FEATURE_COLS].values
X_unknown = unknown[FEATURE_COLS].values

# ---------- Label encoding (shared across all partitions) ----------
# IMPORTANT: fit encoder on train labels only, then transform others.
# Unknown holdout labels will be UNSEEN by the encoder — we handle that separately.
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train[LABEL_COL])
y_val   = label_encoder.transform(val[LABEL_COL])
y_test  = label_encoder.transform(test[LABEL_COL])
# Unknown holdout labels are intentionally NOT encoded — they are OUT-OF-DISTRIBUTION
y_unknown_raw = unknown[LABEL_COL].values  # keep as strings for reference

N_CLASSES = len(label_encoder.classes_)
print(f"Number of known classes: {N_CLASSES}")
print(f"Unknown holdout classes (never seen in training): {sorted(set(y_unknown_raw))}")

# Identify benign class index (needed for Layer 4 AE training)
BENIGN_IDX = int(np.where(label_encoder.classes_ == 'BENIGN')[0][0])
print(f"Benign class index: {BENIGN_IDX}")

Loading parquet splits...
  dev_sample.parquet           (50000, 40)  (2.4 MB)
  unknown_holdout.parquet      (1455, 40)  (0.1 MB)

[DEV MODE] Sub-split of dev_sample:
  train: (34998, 40)
  val:   (7502, 40)
  test:  (7500, 40)

Feature columns: 39
Number of known classes: 29
Unknown holdout classes (never seen in training): ['BACKDOOR_MALWARE', 'BROWSERHIJACKING', 'RECON-PINGSWEEP', 'UPLOADING_ATTACK', 'XSS']
Benign class index: 0


In [ ]:
# ============================================================
# Cell 3: Layer 3 — Random Forest
# ============================================================

print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    class_weight='balanced'  # helps with residual class imbalance
)
rf.fit(X_train, y_train)

# ---------- Evaluate on test set (known classes only) ----------
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1_macro = f1_score(y_test, rf_pred, average='macro', zero_division=0)
rf_f1_weighted = f1_score(y_test, rf_pred, average='weighted', zero_division=0)

print(f"\n--- Random Forest Results ---")
print(f"Accuracy:          {rf_accuracy:.4f}")
print(f"F1 (macro):        {rf_f1_macro:.4f}")
print(f"F1 (weighted):     {rf_f1_weighted:.4f}")

print(f"\nClassification report:")
print(classification_report(
    y_test, rf_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

# ---------- Save RF metrics ----------
rf_metrics = {
    'model': 'RandomForest',
    'mode': 'dev_sample' if USE_DEV_SAMPLE else 'full',
    'accuracy': float(rf_accuracy),
    'f1_macro': float(rf_f1_macro),
    'f1_weighted': float(rf_f1_weighted),
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'n_classes': int(N_CLASSES),
}
with open(os.path.join(RESULTS_L3, 'rf_metrics.json'), 'w') as f:
    json.dump(rf_metrics, f, indent=2)
print(f"\nSaved: {os.path.join(RESULTS_L3, 'rf_metrics.json')}")

Training Random Forest...

--- Random Forest Results ---
Accuracy:          0.7633
F1 (macro):        0.6778
F1 (weighted):     0.7580

Classification report:
                         precision    recall  f1-score   support

                 BENIGN       0.65      0.77      0.71       369
       COMMANDINJECTION       1.00      0.17      0.29         6
 DDOS-ACK_FRAGMENTATION       0.99      1.00      1.00       302
        DDOS-HTTP_FLOOD       0.88      0.70      0.78        30
        DDOS-ICMP_FLOOD       1.00      1.00      1.00       369
DDOS-ICMP_FRAGMENTATION       0.99      0.98      0.99       369
      DDOS-PSHACK_FLOOD       1.00      1.00      1.00       369
       DDOS-RSTFINFLOOD       1.00      1.00      1.00       369
         DDOS-SLOWLORIS       0.71      0.88      0.79        25
DDOS-SYNONYMOUSIP_FLOOD       0.45      0.48      0.47       369
         DDOS-SYN_FLOOD       0.36      0.33      0.34       368
         DDOS-TCP_FLOOD       0.59      0.56      0.57      

In [ ]:
# ============================================================
# Cell 4: Layer 3 — XGBoost
# ============================================================

print("Training XGBoost...")
xgb = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    max_depth=6,
    n_estimators=200,
    tree_method='hist',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
xgb.fit(X_train, y_train)

# ---------- Evaluate on test set ----------
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_f1_macro = f1_score(y_test, xgb_pred, average='macro', zero_division=0)
xgb_f1_weighted = f1_score(y_test, xgb_pred, average='weighted', zero_division=0)

print(f"\n--- XGBoost Results ---")
print(f"Accuracy:          {xgb_accuracy:.4f}")
print(f"F1 (macro):        {xgb_f1_macro:.4f}")
print(f"F1 (weighted):     {xgb_f1_weighted:.4f}")

print(f"\nClassification report:")
print(classification_report(
    y_test, xgb_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

# ---------- Save XGBoost metrics ----------
xgb_metrics = {
    'model': 'XGBoost',
    'mode': 'dev_sample' if USE_DEV_SAMPLE else 'full',
    'accuracy': float(xgb_accuracy),
    'f1_macro': float(xgb_f1_macro),
    'f1_weighted': float(xgb_f1_weighted),
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'n_classes': int(N_CLASSES),
}
with open(os.path.join(RESULTS_L3, 'xgb_metrics.json'), 'w') as f:
    json.dump(xgb_metrics, f, indent=2)
print(f"\nSaved: {os.path.join(RESULTS_L3, 'xgb_metrics.json')}")

Training XGBoost...

--- XGBoost Results ---
Accuracy:          0.7721
F1 (macro):        0.6882
F1 (weighted):     0.7681

Classification report:
                         precision    recall  f1-score   support

                 BENIGN       0.66      0.76      0.71       369
       COMMANDINJECTION       0.50      0.17      0.25         6
 DDOS-ACK_FRAGMENTATION       0.99      0.99      0.99       302
        DDOS-HTTP_FLOOD       0.82      0.77      0.79        30
        DDOS-ICMP_FLOOD       1.00      0.99      1.00       369
DDOS-ICMP_FRAGMENTATION       0.98      0.98      0.98       369
      DDOS-PSHACK_FLOOD       1.00      1.00      1.00       369
       DDOS-RSTFINFLOOD       1.00      1.00      1.00       369
         DDOS-SLOWLORIS       0.85      0.88      0.86        25
DDOS-SYNONYMOUSIP_FLOOD       0.46      0.54      0.49       369
         DDOS-SYN_FLOOD       0.40      0.34      0.37       368
         DDOS-TCP_FLOOD       0.61      0.58      0.60       368
       

In [ ]:
# ============================================================
# Cell 5: Layer 4 — Autoencoder (train on benign ONLY)
# ============================================================

# ---------- Extract benign subsets for AE training and threshold ----------
benign_train_mask = (y_train == BENIGN_IDX)
benign_val_mask = (y_val == BENIGN_IDX)

X_benign_train = X_train[benign_train_mask]
X_benign_val = X_val[benign_val_mask]

print(f"Benign training samples: {len(X_benign_train):,}")
print(f"Benign validation samples: {len(X_benign_val):,}")

# ---------- Build autoencoder ----------
# Architecture adapted from Oscar's notebook: 39 -> 64 -> 32 -> 64 -> 39
input_dim = X_benign_train.shape[1]
inputs = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(inputs)
encoded = Dense(32, activation='relu')(encoded)
decoded = Dense(64, activation='relu')(encoded)
decoded = Dense(input_dim)(decoded)

autoencoder = Model(inputs, decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# ---------- Train ----------
print("\nTraining autoencoder on benign data only...")
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = autoencoder.fit(
    X_benign_train, X_benign_train,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nFinal training loss: {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")

Benign training samples: 1,721
Benign validation samples: 369


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 39)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 39)             │         2,535 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,287 (36.28 KB)

 Trainable params: 9,287 (36.28 KB)

 Non-trainable params: 0 (0.00 B)


Training autoencoder on benign data only...
Epoch 1/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 152ms/step - loss: 2.5354 - val_loss: 2.5475
Epoch 2/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 2.0930 - val_loss: 2.2264
Epoch 3/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 1.8221 - val_loss: 1.9726
Epoch 4/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.5745 - val_loss: 1.7215
Epoch 5/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 1.3337 - val_loss: 1.5031
Epoch 6/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 1.1373 - val_loss: 1.3310
Epoch 7/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.9837 - val_loss: 1.1747
Epoch 8/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.8481 - val_loss: 1.0408
Epoch 9/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.7287 - val_loss: 0.9221
Epoch 10/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.6253 - val_loss: 0.8198
Epoch 11/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.5399 - val_loss: 0.7352
Epoch 12/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s

In [ ]:
# ============================================================
# Cell 6: Layer 4 — Threshold setting (LOCKED RULE)
# ============================================================
# Rule: threshold = 95th percentile of benign VALIDATION reconstruction errors
# This rule was committed before modeling began and is NOT tuned on test data.
# ============================================================

def reconstruction_error(model, X, batch_size=1024):
    """Compute per-sample MSE reconstruction error."""
    preds = model.predict(X, batch_size=batch_size, verbose=0)
    return np.mean(np.square(X - preds), axis=1)

# ---------- Compute errors on benign validation ----------
print("Computing reconstruction errors on benign validation set...")
benign_val_errors = reconstruction_error(autoencoder, X_benign_val)
print(f"  n = {len(benign_val_errors):,}")
print(f"  mean = {benign_val_errors.mean():.6f}")
print(f"  median = {np.median(benign_val_errors):.6f}")
print(f"  95th percentile = {np.percentile(benign_val_errors, 95):.6f}")
print(f"  99th percentile = {np.percentile(benign_val_errors, 99):.6f}")

# ---------- LOCK THE THRESHOLD ----------
AE_THRESHOLD = float(np.percentile(benign_val_errors, 95))
print(f"\n>>> AE_THRESHOLD locked at: {AE_THRESHOLD:.6f} <<<")
print("    (95th percentile of benign validation reconstruction errors)")
print("    This threshold will NOT be changed during test or unknown evaluation.")

# ---------- Compute errors on test set ----------
print("\nComputing reconstruction errors on test set (all classes)...")
test_errors = reconstruction_error(autoencoder, X_test)
print(f"  n = {len(test_errors):,}")
print(f"  mean = {test_errors.mean():.6f}")
print(f"  median = {np.median(test_errors):.6f}")

# Per-class mean reconstruction error on test set
print(f"\nMean reconstruction error per class in test set:")
for cls_idx in range(N_CLASSES):
    mask = (y_test == cls_idx)
    if mask.sum() > 0:
        cls_name = label_encoder.classes_[cls_idx]
        cls_mean = test_errors[mask].mean()
        flag = " *HIGH*" if cls_mean > AE_THRESHOLD else ""
        print(f"  {cls_name:28s}  {cls_mean:.6f}{flag}")

# ---------- Compute errors on unknown holdout ----------
print("\nComputing reconstruction errors on UNKNOWN holdout set...")
unknown_errors = reconstruction_error(autoencoder, X_unknown)
print(f"  n = {len(unknown_errors):,}")
print(f"  mean = {unknown_errors.mean():.6f}")
print(f"  median = {np.median(unknown_errors):.6f}")

# Per-class mean reconstruction error on unknown set
print(f"\nMean reconstruction error per unknown class:")
for cls in sorted(set(y_unknown_raw)):
    mask = (y_unknown_raw == cls)
    cls_mean = unknown_errors[mask].mean()
    flag = " *HIGH*" if cls_mean > AE_THRESHOLD else ""
    print(f"  {cls:28s}  {cls_mean:.6f}{flag}")

Computing reconstruction errors on benign validation set...
  n = 369
  mean = 0.058342
  median = 0.030039
  95th percentile = 0.136996
  99th percentile = 0.499656

>>> AE_THRESHOLD locked at: 0.136996 <<<
    (95th percentile of benign validation reconstruction errors)
    This threshold will NOT be changed during test or unknown evaluation.

Computing reconstruction errors on test set (all classes)...
  n = 7,500
  mean = 0.538018
  median = 0.430439

Mean reconstruction error per class in test set:
  BENIGN                        0.122974
  COMMANDINJECTION              0.073232
  DDOS-ACK_FRAGMENTATION        0.389149 *HIGH*
  DDOS-HTTP_FLOOD               0.682867 *HIGH*
  DDOS-ICMP_FLOOD               0.607301 *HIGH*
  DDOS-ICMP_FRAGMENTATION       0.690945 *HIGH*
  DDOS-PSHACK_FLOOD             0.799179 *HIGH*
  DDOS-RSTFINFLOOD              2.399865 *HIGH*
  DDOS-SLOWLORIS                0.840963 *HIGH*
  DDOS-SYNONYMOUSIP_FLOOD       0.613802 *HIGH*
  DDOS-SYN_FLOOD         

In [ ]:
# ============================================================
# Cell 7: Layer 4 — Sample-level TPR/FPR evaluation
# ============================================================
# Using the locked threshold, compute sample-level detection rates.
#
# Construction of the evaluation set:
#   - Positives (label = 1): all unknown holdout samples (should be flagged)
#   - Negatives (label = 0): benign samples from the test set (should NOT be flagged)
# This gives us a clean binary "unknown vs benign" evaluation.
# ============================================================

# ---------- Build binary evaluation set ----------
benign_test_mask = (y_test == BENIGN_IDX)
benign_test_errors = test_errors[benign_test_mask]

# Positives = unknown holdout samples (label = 1)
# Negatives = benign test samples (label = 0)
eval_errors = np.concatenate([unknown_errors, benign_test_errors])
eval_labels = np.concatenate([
    np.ones(len(unknown_errors)),           # 1 = unknown (anomaly)
    np.zeros(len(benign_test_errors))       # 0 = benign (normal)
])

print(f"Evaluation set:")
print(f"  Unknown holdout samples (positives): {len(unknown_errors):,}")
print(f"  Benign test samples (negatives):     {len(benign_test_errors):,}")
print(f"  Total: {len(eval_errors):,}")

# ---------- Sample-level predictions using LOCKED threshold ----------
ae_predictions = (eval_errors > AE_THRESHOLD).astype(int)

# Confusion matrix components
TP = int(((ae_predictions == 1) & (eval_labels == 1)).sum())
FP = int(((ae_predictions == 1) & (eval_labels == 0)).sum())
TN = int(((ae_predictions == 0) & (eval_labels == 0)).sum())
FN = int(((ae_predictions == 0) & (eval_labels == 1)).sum())

TPR = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # sensitivity / recall
FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0
precision_ae = TP / (TP + FP) if (TP + FP) > 0 else 0.0
accuracy_ae = (TP + TN) / (TP + FP + TN + FN)

print(f"\n--- Layer 4 Autoencoder Open-Set Detection (LOCKED threshold = {AE_THRESHOLD:.4f}) ---")
print(f"  TP (unknown correctly flagged):     {TP:>6,}")
print(f"  FN (unknown missed):                {FN:>6,}")
print(f"  TN (benign correctly kept):         {TN:>6,}")
print(f"  FP (benign incorrectly flagged):    {FP:>6,}")
print(f"")
print(f"  TPR (sensitivity): {TPR:.4f}")
print(f"  FPR (false positive rate): {FPR:.4f}")
print(f"  Precision: {precision_ae:.4f}")
print(f"  Accuracy:  {accuracy_ae:.4f}")
print(f"")
print(f"  Target: TPR >= 0.95, FPR < 0.05")
print(f"  Met: TPR={'✓' if TPR >= 0.95 else '✗'}  FPR={'✓' if FPR < 0.05 else '✗'}")

# ---------- ROC-AUC (threshold-independent view) ----------
roc_auc = roc_auc_score(eval_labels, eval_errors)
print(f"\n  ROC-AUC (threshold-independent): {roc_auc:.4f}")

# ---------- Per-class detection rate on unknowns ----------
print(f"\nPer-class detection rate on unknown holdout:")
for cls in sorted(set(y_unknown_raw)):
    mask = (y_unknown_raw == cls)
    n_total = mask.sum()
    n_detected = ((unknown_errors[mask] > AE_THRESHOLD).sum())
    rate = n_detected / n_total
    print(f"  {cls:28s}  {n_detected:>4}/{n_total:<4} = {rate:.1%}")

# ---------- Save Layer 4 metrics ----------
l4_metrics = {
    'layer': 4,
    'model': 'Autoencoder',
    'mode': 'dev_sample' if USE_DEV_SAMPLE else 'full',
    'threshold': AE_THRESHOLD,
    'threshold_rule': '95th percentile of benign validation reconstruction errors',
    'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
    'TPR': float(TPR),
    'FPR': float(FPR),
    'precision': float(precision_ae),
    'accuracy': float(accuracy_ae),
    'roc_auc': float(roc_auc),
    'n_positive': int(eval_labels.sum()),
    'n_negative': int(len(eval_labels) - eval_labels.sum()),
}
with open(os.path.join(RESULTS_L4, 'ae_metrics.json'), 'w') as f:
    json.dump(l4_metrics, f, indent=2)
print(f"\nSaved: {os.path.join(RESULTS_L4, 'ae_metrics.json')}")

Evaluation set:
  Unknown holdout samples (positives): 1,455
  Benign test samples (negatives):     369
  Total: 1,824

--- Layer 4 Autoencoder Open-Set Detection (LOCKED threshold = 0.1370) ---
  TP (unknown correctly flagged):        361
  FN (unknown missed):                 1,094
  TN (benign correctly kept):            345
  FP (benign incorrectly flagged):        24

  TPR (sensitivity): 0.2481
  FPR (false positive rate): 0.0650
  Precision: 0.9377
  Accuracy:  0.3871

  Target: TPR >= 0.95, FPR < 0.05
  Met: TPR=✗  FPR=✗

  ROC-AUC (threshold-independent): 0.7606

Per-class detection rate on unknown holdout:
  BACKDOOR_MALWARE                63/275  = 22.9%
  BROWSERHIJACKING               173/508  = 34.1%
  RECON-PINGSWEEP                 43/182  = 23.6%
  UPLOADING_ATTACK                21/122  = 17.2%
  XSS                             61/368  = 16.6%

Saved: /content/drive/MyDrive/Capstone_IoT/Results/Layer4/ae_metrics.json


In [ ]:
# ============================================================
# Cell 8: Layer 5 — Decision Fusion
# ============================================================
# Fusion formula adapted from Oscar's capstone.ipynb:
#   risk_score = w1*supervised_uncertainty + w2*ae_anomaly_binary + w3*kmeans_anomaly_binary
#
# Changes from Oscar's original:
#   1. Uses XGBoost uncertainty (best Layer 3 model) instead of RF
#   2. Evaluates on test + unknown_holdout separately (not full dataset)
#   3. Compares three conditions: supervised-only, AE-only, fused
#   4. Uses the locked AE threshold (not re-tuned)
# ============================================================

# ---------- Fusion weights (can be tuned on val, locked before test) ----------
W_SUPERVISED = 0.4
W_AE = 0.3
W_KMEANS = 0.3

print(f"Fusion weights: supervised={W_SUPERVISED}, AE={W_AE}, KMeans={W_KMEANS}")

# ---------- Helper: compute supervised uncertainty ----------
def supervised_uncertainty(model, X):
    """Return 1 - max_class_probability for each sample (higher = less confident)."""
    proba = model.predict_proba(X)
    return 1.0 - proba.max(axis=1)

# ---------- KMeans on benign training data (for distance-based anomaly) ----------
# Adapted from Oscar's notebook — 1-cluster KMeans centered on benign
print("\nFitting 1-cluster KMeans on benign training data...")
kmeans = KMeans(n_clusters=1, n_init=10, random_state=RANDOM_SEED)
kmeans.fit(X_benign_train)

def kmeans_distance(X):
    """Distance from each sample to the benign cluster centroid."""
    return np.linalg.norm(X - kmeans.cluster_centers_[0], axis=1)

# ---------- Lock KMeans threshold on benign validation (same rule as AE) ----------
benign_val_kmeans_dist = kmeans_distance(X_benign_val)
KMEANS_THRESHOLD = float(np.percentile(benign_val_kmeans_dist, 95))
print(f"KMeans threshold locked at: {KMEANS_THRESHOLD:.4f}")
print("  (95th percentile of benign validation distances to centroid)")

# ---------- Build binary evaluation set: unknown vs benign_test ----------
# Same structure as Cell 7 — unknown holdout (positives) vs benign test (negatives)
eval_X = np.concatenate([X_unknown, X_test[benign_test_mask]])
eval_labels = np.concatenate([
    np.ones(len(X_unknown)),
    np.zeros(benign_test_mask.sum())
])
print(f"\nEvaluation set: {int(eval_labels.sum())} positives + {int((1-eval_labels).sum())} negatives = {len(eval_labels)}")

# ---------- Condition 1: Supervised-only (XGBoost uncertainty) ----------
sup_scores = supervised_uncertainty(xgb, eval_X)
sup_threshold = float(np.percentile(supervised_uncertainty(xgb, X_benign_val), 95))
print(f"\nSupervised uncertainty threshold (95th percentile benign val): {sup_threshold:.4f}")
sup_pred = (sup_scores > sup_threshold).astype(int)

def compute_metrics(pred, labels, name):
    TP = int(((pred == 1) & (labels == 1)).sum())
    FP = int(((pred == 1) & (labels == 0)).sum())
    TN = int(((pred == 0) & (labels == 0)).sum())
    FN = int(((pred == 0) & (labels == 1)).sum())
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    return {'name': name, 'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
            'TPR': TPR, 'FPR': FPR, 'precision': precision}

sup_metrics = compute_metrics(sup_pred, eval_labels, "Supervised-only (XGBoost uncertainty)")

# ---------- Condition 2: AE-only (already computed in Cell 7) ----------
ae_scores_eval = np.concatenate([unknown_errors, test_errors[benign_test_mask]])
ae_pred = (ae_scores_eval > AE_THRESHOLD).astype(int)
ae_metrics_eval = compute_metrics(ae_pred, eval_labels, "AE-only")

# ---------- Condition 3: KMeans-only ----------
km_scores = kmeans_distance(eval_X)
km_pred = (km_scores > KMEANS_THRESHOLD).astype(int)
km_metrics = compute_metrics(km_pred, eval_labels, "KMeans-only")

# ---------- Condition 4: FUSED (the paper's central claim) ----------
# Normalize each score to [0, 1] before combining
def minmax_normalize(x):
    xmin, xmax = x.min(), x.max()
    if xmax - xmin < 1e-12:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

sup_norm = minmax_normalize(sup_scores)
ae_norm = minmax_normalize(ae_scores_eval)
km_norm = minmax_normalize(km_scores)

risk_score = W_SUPERVISED * sup_norm + W_AE * ae_norm + W_KMEANS * km_norm
risk_score = np.clip(risk_score, 0, 1)

# Lock fusion threshold using the same rule: 95th percentile on benign val
sup_val_norm = minmax_normalize(supervised_uncertainty(xgb, X_benign_val))
ae_val = reconstruction_error(autoencoder, X_benign_val)
ae_val_norm = minmax_normalize(ae_val)
km_val_norm = minmax_normalize(kmeans_distance(X_benign_val))
benign_val_risk = W_SUPERVISED * sup_val_norm + W_AE * ae_val_norm + W_KMEANS * km_val_norm
FUSION_THRESHOLD = float(np.percentile(benign_val_risk, 95))
print(f"\nFusion threshold locked at: {FUSION_THRESHOLD:.4f}")

fused_pred = (risk_score > FUSION_THRESHOLD).astype(int)
fused_metrics = compute_metrics(fused_pred, eval_labels, "FUSED (Layer 5)")

# ---------- Compare all four conditions ----------
print(f"\n{'='*75}")
print(f"{'Condition':<42} {'TPR':>8} {'FPR':>8} {'Prec':>8}")
print(f"{'='*75}")
for m in [sup_metrics, ae_metrics_eval, km_metrics, fused_metrics]:
    print(f"{m['name']:<42} {m['TPR']:>8.4f} {m['FPR']:>8.4f} {m['precision']:>8.4f}")
print(f"{'='*75}")
print(f"Target: TPR >= 0.95, FPR < 0.05")

# ---------- ROC-AUC for each condition ----------
print(f"\nROC-AUC (threshold-independent):")
print(f"  Supervised-only: {roc_auc_score(eval_labels, sup_scores):.4f}")
print(f"  AE-only:         {roc_auc_score(eval_labels, ae_scores_eval):.4f}")
print(f"  KMeans-only:     {roc_auc_score(eval_labels, km_scores):.4f}")
print(f"  FUSED:           {roc_auc_score(eval_labels, risk_score):.4f}")

# ---------- Save all Layer 5 metrics ----------
l5_metrics = {
    'layer': 5,
    'mode': 'dev_sample' if USE_DEV_SAMPLE else 'full',
    'fusion_weights': {'supervised': W_SUPERVISED, 'ae': W_AE, 'kmeans': W_KMEANS},
    'ae_threshold': AE_THRESHOLD,
    'kmeans_threshold': KMEANS_THRESHOLD,
    'supervised_threshold': sup_threshold,
    'fusion_threshold': FUSION_THRESHOLD,
    'conditions': {
        'supervised_only': sup_metrics,
        'ae_only': ae_metrics_eval,
        'kmeans_only': km_metrics,
        'fused': fused_metrics,
    },
    'roc_auc': {
        'supervised': float(roc_auc_score(eval_labels, sup_scores)),
        'ae': float(roc_auc_score(eval_labels, ae_scores_eval)),
        'kmeans': float(roc_auc_score(eval_labels, km_scores)),
        'fused': float(roc_auc_score(eval_labels, risk_score)),
    }
}
with open(os.path.join(RESULTS_L5, 'fusion_metrics.json'), 'w') as f:
    json.dump(l5_metrics, f, indent=2, default=str)
print(f"\nSaved: {os.path.join(RESULTS_L5, 'fusion_metrics.json')}")

Fusion weights: supervised=0.4, AE=0.3, KMeans=0.3

Fitting 1-cluster KMeans on benign training data...
KMeans threshold locked at: 9.8249
  (95th percentile of benign validation distances to centroid)

Evaluation set: 1455 positives + 369 negatives = 1824

Supervised uncertainty threshold (95th percentile benign val): 0.5908

Fusion threshold locked at: 0.3547

Condition                                       TPR      FPR     Prec
Supervised-only (XGBoost uncertainty)        0.0880   0.0488   0.8767
AE-only                                      0.2481   0.0650   0.9377
KMeans-only                                  0.1533   0.0894   0.8711
FUSED (Layer 5)                              0.0811   0.0352   0.9008
Target: TPR >= 0.95, FPR < 0.05

ROC-AUC (threshold-independent):
  Supervised-only: 0.6470
  AE-only:         0.7606
  KMeans-only:     0.5972
  FUSED:           0.6619

Saved: /content/drive/MyDrive/Capstone_IoT/Results/Layer5/fusion_metrics.json
